In [2]:
import subprocess
subprocess.run(["pip", "install", "-q", "groq"], check=True)

from google.colab import drive
import pandas as pd
import torch
from groq import Groq
from google.colab import userdata

#folder_path = "/content/drive/My Drive/codepath/AI201/data"
drive.mount('/content/drive')
CSV_PATH = "data/hw3_dating_labeled.csv"
df = pd.read_csv(CSV_PATH)

groq_client = Groq(api_key=GROQ_API_KEY)

print(f"PyTorch {torch.__version__} | GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[!] No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU.")

Mounted at /content/drive
PyTorch 2.11.0+cu128 | GPU available: True
GPU: Tesla T4


# TakeMeter — Fine-Tuning Starter Notebook
### AI201 · Project 3

This notebook walks you through fine-tuning a text classifier on your annotated dataset and comparing it to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Runs the fine-tuning pipeline with DistilBERT
- Computes evaluation metrics and generates a confusion matrix
- Runs the Groq baseline and compares both models

**What you do (the actual work):**
- Collect and annotate your 200+ examples (done before opening this notebook)
- Define your label map and upload your CSV
- Write your Groq classification prompt using your label definitions
- Analyze the output and write your evaluation report

---
**Before you start:** Make sure you are using a T4 GPU runtime.  
Go to **Runtime → Change runtime type → T4 GPU**, then click Save.

---
## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map.  
Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).

In [3]:
from google.colab import drive
import pandas as pd
import torch
from groq import Groq
from google.colab import userdata

drive.mount('/content/drive')
#folder_path = "/content/drive/My Drive/codepath/AI201/data"

CSV_PATH = "/content/drive/My Drive/codepath/AI201/data/hw3_dating_labeled.csv"
df = pd.read_csv(CSV_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU is available:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU is available: Tesla T4


In [4]:
"""
takemeter_full.py
=================
AI201 Project 3 -- TakeMeter | r/dating classifier
Combined Milestone 4 + Milestone 5

Runs in a single Google Colab session (T4 GPU recommended).
Sections run top-to-bottom:
  1. Installs + Imports
  2. Config: label map, paths, model name
  3. Data:   load CSV, split, tokenize
  4. Baseline: Groq zero-shot evaluation on test set
  5. Fine-tuning: DistilBERT training on train set
  6. Evaluation: fine-tuned model metrics on test set
  7. Comparison + Export: side-by-side results, JSON, confusion matrix

Before running:
  - Set runtime to T4 GPU (Runtime -> Change runtime type -> T4 GPU)
  - Add GROQ_API_KEY to Colab Secrets (key icon in left sidebar)
  - Clone your repo and cd into it so data/hw3_dating_labeled.csv is reachable:
      !git clone https://github.com/<your-username>/<your-repo>.git
      %cd <your-repo>
"""

# ==================================================================
# SECTION 1 -- Installs + Imports
# ==================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "groq"], check=True)

import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from groq import Groq
from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from datasets import Dataset

print("Imports complete.")

Imports complete.


In [5]:



# ==================================================================
# SECTION 1.5 -- Config
# Edit LABEL_MAP and CSV_PATH here if anything changes.
# ==================================================================

LABEL_MAP = {
    "vent":       0,
    "advice":     1,
    "discussion": 2,
}
NUM_LABELS  = len(LABEL_MAP)
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
MODEL_NAME  = "distilbert-base-uncased"
CSV_PATH    = "data/hw3_dating_labeled.csv"

print(f"\nLabel map : {LABEL_MAP}")
print(f"Model     : {MODEL_NAME}")
print(f"CSV path  : {CSV_PATH}")


Label map : {'vent': 0, 'advice': 1, 'discussion': 2}
Model     : distilbert-base-uncased
CSV path  : data/hw3_dating_labeled.csv


---
## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.

In [7]:
# ==================================================================
# SECTION 2 -- Data: load, split, tokenize
# ==================================================================

# -- Load CSV ------------------------------------------------------

print(f"\nLoaded {len(df)} examples from {CSV_PATH}")
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts())

unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    raise ValueError(f"Unknown labels in CSV: {unknown}. Update LABEL_MAP in Section 2.")
print("[OK] All labels match LABEL_MAP")

df["label_id"] = df["label"].map(LABEL_MAP)

# -- Train / Val / Test split (70 / 15 / 15, stratified) ----------
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain : {len(train_df)} examples")
print(f"Val   : {len(val_df)}  examples")
print(f"Test  : {len(test_df)}  examples")
print("\nTest label distribution:")
print(test_df["label"].value_counts())
print("\n[OK] Test set locked -- do not re-run this section after viewing results.")

# -- Tokenize ------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

def make_hf_dataset(dataframe):
    ds = Dataset.from_dict({
        "text":  dataframe["text"].tolist(),
        "label": dataframe["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

train_dataset = make_hf_dataset(train_df)
val_dataset   = make_hf_dataset(val_df)
test_dataset  = make_hf_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"\n[OK] Tokenization complete.")
print(f"     Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")



Loaded 789 examples from data/hw3_dating_labeled.csv
Columns: ['text', 'label']

Label distribution:
label
discussion    283
vent          267
advice        239
Name: count, dtype: int64
[OK] All labels match LABEL_MAP

Train : 552 examples
Val   : 118  examples
Test  : 119  examples

Test label distribution:
label
discussion    43
vent          40
advice        36
Name: count, dtype: int64

[OK] Test set locked -- do not re-run this section after viewing results.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Map:   0%|          | 0/119 [00:00<?, ? examples/s]


[OK] Tokenization complete.
     Train: 552 | Val: 118 | Test: 119


---
## Section 5: Baseline Classifier (Groq)

Runs your zero-shot baseline using `llama-3.3-70b-versatile`.  
You need to write the classification prompt using your label definitions.

In [8]:





# ==================================================================
# SECTION 5 -- Groq Baseline (zero-shot on test set)
# Defines: baseline_preds, bl_accuracy, bl_macro_f1, bl_report
# ==================================================================

#GROQ_API_KEY = userdata.get("GROQ_API_KEY")
#assert GROQ_API_KEY, (
    #"GROQ_API_KEY not found.\n"
    #"Click the key icon in the left sidebar -> Add secret -> name: GROQ_API_KEY"
#)
#groq_client = Groq(api_key=GROQ_API_KEY)
#print("[OK] Groq client ready")

SYSTEM_PROMPT = """
You are classifying posts from r/dating, a Reddit community for venting, discussing, and learning about dating.

Assign each post to exactly one of the following three categories:

vent
Definition: The post primarily expresses frustration, sadness, anger, or emotional relief about a dating experience. The writer's goal is to be heard or validated, not to receive specific guidance or invite debate.
Example: "I'm so tired of guys who string you along just for sex. It's manipulative and hurtful."

advice
Definition: The post either requests concrete guidance on a specific personal situation, OR offers actionable tips and lessons directly to other daters. The defining feature is an explicit transaction of practical knowledge.
Example: "Movie, then dinner -- not dinner and a movie. You go to the movie first and have an immediate conversation topic walking out."

discussion
Definition: The post poses a question, shares an observation, or tells a story primarily to invite broader community conversation. There is no specific personal crisis and no practical tip being transacted.
Example: "Would you use a dating app where you had to listen to a 15-minute podcast about someone before matching?"

Decision rules for ambiguous posts:
- If a post vents AND ends with an explicit verdict-seeking question ("did I dodge a bullet?", "should I leave?"), label it: advice
- If a post vents AND the question is rhetorical ("why do people do this?"), label it: vent
- If a post gives a directive ("do X") AND includes reasoning readers can use, label it: advice
- If a post shares a frustrated observation AND invites others to weigh in, label it: discussion

Respond with ONLY the label name. No explanation. No punctuation. No capitalization variants.
Valid responses: vent | advice | discussion
""".strip()


def classify_with_groq(text, retries=3):
    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": f"Classify this post:\n\n{text[:1200]}"},
                ],
                temperature=0,
                max_tokens=10,
            )
            raw = response.choices[0].message.content.strip().lower()
            for label in sorted(LABEL_MAP, key=len, reverse=True):
                if label in raw:
                    return label
            print(f"  [!] Unparseable response: '{raw}'")
            return None
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"  Rate limited -- waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  API error: {e}")
                return None
    return None


print(f"\nRunning Groq baseline on {len(test_df)} test examples...")
print("(expect 3-8 minutes on free tier)\n")

baseline_preds = []
for i, row in test_df.iterrows():
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    pos = test_df.index.get_loc(i)
    if (pos + 1) % 10 == 0:
        print(f"  {pos + 1}/{len(test_df)} classified...")
    time.sleep(0.7)

print("\n[OK] Baseline classification complete.")


Running Groq baseline on 119 test examples...
(expect 3-8 minutes on free tier)

  10/119 classified...
  20/119 classified...
  30/119 classified...
  40/119 classified...
  50/119 classified...
  60/119 classified...
  70/119 classified...
  80/119 classified...
  90/119 classified...
  100/119 classified...
  110/119 classified...

[OK] Baseline classification complete.


In [9]:
# -- Baseline metrics ----------------------------------------------
bl_label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]

unparseable = [p for p in baseline_preds if p is None]
print(f"Unparseable responses: {len(unparseable)} / {len(test_df)}")
if len(unparseable) / len(test_df) > 0.10:
    print("[!] >10% unparseable -- consider tightening the system prompt.")

bl_valid = [
    (LABEL_MAP[pred], true_id)
    for pred, true_id in zip(baseline_preds, test_df["label_id"])
    if pred is not None
]
bl_pred_ids = [p for p, _ in bl_valid]
bl_true_ids = [t for _, t in bl_valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
bl_macro_f1 = f1_score(bl_true_ids, bl_pred_ids, average="macro", zero_division=0)
bl_report   = classification_report(
    bl_true_ids, bl_pred_ids,
    target_names=bl_label_names,
    output_dict=True,
    zero_division=0,
)

Unparseable responses: 0 / 119


In [10]:
print("\n" + "="*52)
print("  BASELINE RESULTS  (Groq llama-3.3-70b-versatile)")
print("="*52)
print(f"  Overall accuracy : {bl_accuracy:.3f}")
print(f"  Macro F1         : {bl_macro_f1:.3f}")
print(f"  Evaluated on     : {len(bl_valid)} / {len(test_df)} examples")
print()
print(classification_report(
    bl_true_ids, bl_pred_ids,
    target_names=bl_label_names,
    zero_division=0, digits=3,
))
print("="*52)

# -- Save baseline confusion matrix --------------------------------
bl_cm = confusion_matrix(bl_true_ids, bl_pred_ids)
bl_disp = ConfusionMatrixDisplay(confusion_matrix=bl_cm, display_labels=bl_label_names)
fig, ax = plt.subplots(figsize=(6, 5))
bl_disp.plot(ax=ax, cmap="Oranges", colorbar=False)
ax.set_title("Groq Baseline -- Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("baseline_confusion_matrix.png", dpi=150)
plt.show()
print("[OK] Saved: baseline_confusion_matrix.png")

# -- Wrong predictions for baseline --------------------------------
bl_wrong = [
    i for i, (pred, true_id) in enumerate(zip(baseline_preds, test_df["label_id"]))
    if pred is not None and LABEL_MAP[pred] != true_id
]
print(f"\nBaseline wrong predictions: {len(bl_wrong)} / {len(bl_valid)}")
print("\nFirst 10 baseline errors:")
for rank, i in enumerate(bl_wrong[:10]):
    text     = test_df.iloc[i]["text"]
    true_lbl = ID_TO_LABEL[test_df.iloc[i]["label_id"]]
    pred_lbl = baseline_preds[i]
    print(f"  [{rank+1:02d}] TRUE={true_lbl:<12} PRED={pred_lbl}")
    print(f"        {text[:160].strip()}")
    print()

print("[OK] Baseline complete. Proceeding to fine-tuning.\n")



  BASELINE RESULTS  (Groq llama-3.3-70b-versatile)
  Overall accuracy : 0.706
  Macro F1         : 0.710
  Evaluated on     : 119 / 119 examples

              precision    recall  f1-score   support

        vent      0.718     0.700     0.709        40
      advice      0.806     0.694     0.746        36
  discussion      0.633     0.721     0.674        43

    accuracy                          0.706       119
   macro avg      0.719     0.705     0.710       119
weighted avg      0.714     0.706     0.708       119

[OK] Saved: baseline_confusion_matrix.png

Baseline wrong predictions: 35 / 119

First 10 baseline errors:
  [01] TRUE=advice       PRED=discussion
        Others may come and go, your relationship with yourself is FOREVER  taken from someone else's comment earlier today.

I was just thinking that's kind of funny a

  [02] TRUE=discussion   PRED=vent
        I went on my 4th date with a girl from Tinder which ended pretty bad. LENGTHY STORY FROM TONIGHT'S DATE.   


S

---
## Section 3: Fine-Tune Your Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on your training data.  
Training runs for 3 epochs and takes **5–15 minutes** on a T4 GPU.

> **Hyperparameter note:** The defaults below work well for datasets of 100–500 examples.  
> If you change any values, note what you changed and why in your README.

In [11]:
# ==================================================================
# SECTION 3 -- Fine-tuning (DistilBERT)
# ==================================================================

# -- Load model ----------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"[OK] Model loaded: {MODEL_NAME}")
print(f"     Labels: {NUM_LABELS} -> {ID_TO_LABEL}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"     Device: {device}")
if device == "cpu":
    print("[!] No GPU -- training will be very slow.")

# -- Metrics helper ------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

# -- Training arguments --------------------------------------------
# Hyperparameter notes:
#   num_train_epochs=3     : standard for small datasets; more risks overfitting
#   learning_rate=2e-5     : standard BERT fine-tuning starting point
#   per_device_train_batch_size=16 : fits T4 GPU at max_length=256
#   warmup_steps=30        : ~18% of epoch 1; prevents early gradient distortion
#   weight_decay=0.01      : L2 regularization; helps on small data
#   load_best_model_at_end : restores best val-accuracy checkpoint after training

training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=30,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"\nStarting fine-tuning on {len(train_dataset)} training examples...")
print("Expected time: 5-15 minutes on T4 GPU\n")

trainer.train()

print("\n[OK] Fine-tuning complete.")
print("Best checkpoint restored (load_best_model_at_end=True).")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[OK] Model loaded: distilbert-base-uncased
     Labels: 3 -> {0: 'vent', 1: 'advice', 2: 'discussion'}
     Device: cuda

Starting fine-tuning on 552 training examples...
Expected time: 5-15 minutes on T4 GPU



Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.094638,1.091425,0.364407,0.192361
2,1.067906,1.079509,0.440678,0.360441
3,1.024206,1.066021,0.432203,0.423289


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[OK] Fine-tuning complete.
Best checkpoint restored (load_best_model_at_end=True).


---
## Section 4: Evaluate Fine-Tuned Model on Test Set

Runs inference on your locked test set and generates metrics and a confusion matrix.  
These numbers go directly into your evaluation report.

In [12]:










# ==================================================================
# SECTION 4 -- Evaluate fine-tuned model on test set
# ==================================================================

print("\nRunning inference on locked test set...")
ft_output   = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids
ft_probs    = F.softmax(torch.tensor(ft_output.predictions), dim=-1).numpy()

ft_label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]

ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
ft_macro_f1 = f1_score(ft_true_ids, ft_pred_ids, average="macro", zero_division=0)
ft_report   = classification_report(
    ft_true_ids, ft_pred_ids,
    target_names=ft_label_names,
    output_dict=True,
    zero_division=0,
)

print("\n" + "="*54)
print("  FINE-TUNED MODEL RESULTS  (DistilBERT, 3 epochs)")
print("="*54)
print(f"  Overall accuracy : {ft_accuracy:.3f}")
print(f"  Macro F1         : {ft_macro_f1:.3f}")
print(f"  (planning.md threshold: 0.72 min / 0.78 target)")
print()
print(classification_report(
    ft_true_ids, ft_pred_ids,
    target_names=ft_label_names,
    zero_division=0, digits=3,
))

if ft_macro_f1 < 0.72:
    print("[!] Macro F1 below 0.72 minimum threshold.")
    print("    Review wrong predictions below before writing your report.")
elif ft_macro_f1 >= 0.78:
    print("[OK] Macro F1 meets the 0.78 target threshold.")
else:
    print("[OK] Macro F1 meets the 0.72 minimum threshold.")
    print("    Does not yet reach 0.78 target -- note this in your report.")

# -- Confusion matrix (fine-tuned) ---------------------------------
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ft_label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned DistilBERT -- Confusion Matrix (Test Set)\nr/dating: vent / advice / discussion")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("[OK] Saved: confusion_matrix.png\n")

# -- Confusion matrix as markdown table (for README) ---------------
print("Confusion matrix (paste into README.md):\n")
header  = "| True \\ Pred |" + "".join(f" {n:>12} |" for n in ft_label_names)
divider = "|" + "---|" * (NUM_LABELS + 1)
print(header)
print(divider)
for i, true_name in enumerate(ft_label_names):
    row = f"| **{true_name:<10}** |" + "".join(f" {cm[i,j]:>12} |" for j in range(NUM_LABELS))
    print(row)

print()
cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)
worst_row, worst_col = np.unravel_index(cm_copy.argmax(), cm_copy.shape)
print(
    f"Largest off-diagonal cell: "
    f"true={ft_label_names[worst_row]} -> predicted={ft_label_names[worst_col]} "
    f"({cm_copy[worst_row, worst_col]} cases)"
)

# -- All wrong predictions -----------------------------------------
wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"\nWrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}")
print(f"Test-set accuracy: {ft_accuracy:.3f}\n")

print("--- ALL WRONG PREDICTIONS -------------------------------------------")
for rank, idx in enumerate(wrong_idx):
    text      = test_df.iloc[idx]["text"]
    true_lbl  = ID_TO_LABEL[ft_true_ids[idx]]
    pred_lbl  = ID_TO_LABEL[ft_pred_ids[idx]]
    pred_conf = ft_probs[idx][ft_pred_ids[idx]]
    true_conf = ft_probs[idx][ft_true_ids[idx]]
    print(
        f"[{rank+1:02d}] TRUE={true_lbl:<12} PRED={pred_lbl:<12} "
        f"pred_conf={pred_conf:.2f}  true_conf={true_conf:.2f}"
    )
    print(f"      {text[:220].strip()}")
    print()

print("--- PICKING YOUR 3 DEEP-DIVE CASES FOR THE README ------------------")
print("""
For each of the 3 cases you select, record:
  1. The full post text (or first 300 chars if long)
  2. True label  vs.  predicted label
  3. Prediction confidence
  4. Why would the model make this mistake? (refer to planning.md decision rules)
  5. Is this error systematic (multiple similar cases) or one-off?

Good candidates:
  * One high-confidence wrong prediction (pred_conf > 0.80)
  * One from the most-confused label pair (shown above)
  * One that matches an edge case from planning.md
""")




Running inference on locked test set...



  FINE-TUNED MODEL RESULTS  (DistilBERT, 3 epochs)
  Overall accuracy : 0.403
  Macro F1         : 0.325
  (planning.md threshold: 0.72 min / 0.78 target)

              precision    recall  f1-score   support

        vent      0.750     0.075     0.136        40
      advice      0.391     0.250     0.305        36
  discussion      0.391     0.837     0.533        43

    accuracy                          0.403       119
   macro avg      0.511     0.387     0.325       119
weighted avg      0.512     0.403     0.331       119

[!] Macro F1 below 0.72 minimum threshold.
    Review wrong predictions below before writing your report.
[OK] Saved: confusion_matrix.png

Confusion matrix (paste into README.md):

| True \ Pred |         vent |       advice |   discussion |
|---|---|---|---|
| **vent      ** |            3 |            7 |           30 |
| **advice    ** |            1 |            9 |           26 |
| **discussion** |            0 |            7 |           36 |

Largest 

---
## Section 6: Compare Results and Export

Side-by-side comparison of both models.  
Download the output files and commit them to your GitHub repo.

In [13]:

# ==================================================================
# SECTION 6 -- Comparison table + Export
# ==================================================================

label_names_for_table = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]

print("=" * 62)
print("  RESULTS COMPARISON -- Final")
print("=" * 62)
print(f"  {'Metric':<32} {'Baseline':>10} {'Fine-tuned':>12}")
print("  " + "-" * 56)
print(f"  {'Accuracy':<32} {bl_accuracy:>10.3f} {ft_accuracy:>12.3f}")
print(f"  {'Macro F1':<32} {bl_macro_f1:>10.3f} {ft_macro_f1:>12.3f}")
print()

for lbl in label_names_for_table:
    bl_f1 = bl_report.get(lbl, {}).get("f1-score", float("nan"))
    ft_f1 = ft_report.get(lbl, {}).get("f1-score", float("nan"))
    delta = ft_f1 - bl_f1 if (bl_f1 == bl_f1 and ft_f1 == ft_f1) else float("nan")
    delta_str = f"{delta:+.3f}" if delta == delta else "n/a"
    print(f"  {'F1 ' + lbl:<32} {bl_f1:>10.3f} {ft_f1:>12.3f}   ({delta_str})")

print()
acc_delta = ft_accuracy - bl_accuracy
f1_delta  = ft_macro_f1 - bl_macro_f1
print(f"  Accuracy delta : {acc_delta:+.3f}  ({'^ improvement' if acc_delta >= 0 else 'v regression'})")
print(f"  Macro F1 delta : {f1_delta:+.3f}  ({'^ improvement' if f1_delta >= 0 else 'v regression'})")
print()

if f1_delta >= 0.05:
    print("  [OK] Fine-tuning beats baseline by >= 0.05 macro F1 (planning.md criterion met).")
elif f1_delta >= 0:
    print("  [!] Fine-tuning improved over baseline but by < 0.05 macro F1.")
    print("      Discuss this gap honestly in your README.")
else:
    print("  [!] Fine-tuned model did NOT outperform baseline on macro F1.")
    print("      Investigate: label leakage? class imbalance? annotation noise?")

print("=" * 62)

# -- Export evaluation_results.json --------------------------------
results = {
    "model_finetuned": MODEL_NAME,
    "model_baseline":  "llama-3.3-70b-versatile (zero-shot)",
    "test_set_size":   len(test_df),
    "label_map":       LABEL_MAP,

    "baseline": {
        "accuracy":        round(bl_accuracy, 4),
        "macro_f1":        round(bl_macro_f1, 4),
        "evaluated_on":    len(bl_valid),
        "unparseable":     len(unparseable),
        "per_label": {
            lbl: {
                "precision": round(float(bl_report[lbl]["precision"]), 3),
                "recall":    round(float(bl_report[lbl]["recall"]),    3),
                "f1":        round(float(bl_report[lbl]["f1-score"]),  3),
            }
            for lbl in label_names_for_table
        },
        "confusion_matrix": bl_cm.tolist(),
    },

    "finetuned": {
        "accuracy":       round(ft_accuracy, 4),
        "macro_f1":       round(ft_macro_f1, 4),
        "epochs":         3,
        "learning_rate":  2e-5,
        "batch_size":     16,
        "warmup_steps":   30,
        "weight_decay":   0.01,
        "max_seq_length": 256,
        "per_label": {
            lbl: {
                "precision": round(float(ft_report[lbl]["precision"]), 3),
                "recall":    round(float(ft_report[lbl]["recall"]),    3),
                "f1":        round(float(ft_report[lbl]["f1-score"]),  3),
            }
            for lbl in label_names_for_table
        },
        "confusion_matrix": cm.tolist(),
    },

    "delta": {
        "accuracy": round(acc_delta, 4),
        "macro_f1": round(f1_delta,  4),
    },

    "planning_md_thresholds": {
        "macro_f1_minimum":              0.72,
        "macro_f1_target":               0.78,
        "min_improvement_over_baseline": 0.05,
        "minimum_met":                   bool(ft_macro_f1 >= 0.72),
        "target_met":                    bool(ft_macro_f1 >= 0.78),
        "improvement_criterion_met":     bool(f1_delta >= 0.05),
    },
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n[OK] Saved: evaluation_results.json")
print("[OK] Saved: confusion_matrix.png")
print("[OK] Saved: baseline_confusion_matrix.png")
print()
print("Download all three via: Files panel (left sidebar) -> right-click -> Download")
print("Commit all three to your GitHub repo.")

  RESULTS COMPARISON -- Final
  Metric                             Baseline   Fine-tuned
  --------------------------------------------------------
  Accuracy                              0.706        0.403
  Macro F1                              0.710        0.325

  F1 vent                               0.709        0.136   (-0.572)
  F1 advice                             0.746        0.305   (-0.441)
  F1 discussion                         0.674        0.533   (-0.141)

  Accuracy delta : -0.303  (v regression)
  Macro F1 delta : -0.385  (v regression)

  [!] Fine-tuned model did NOT outperform baseline on macro F1.
      Investigate: label leakage? class imbalance? annotation noise?

[OK] Saved: evaluation_results.json
[OK] Saved: confusion_matrix.png
[OK] Saved: baseline_confusion_matrix.png

Download all three via: Files panel (left sidebar) -> right-click -> Download
Commit all three to your GitHub repo.


# FINISHED